# Fish-model measurement accuracy — publication figures

Two exports, same `|`-delimited schema (the geometry columns are JSON and contain commas):

- `data/all.csv` — the 2026-08-26 calibration-repair session (`HANDOFF.md`, beside this
  notebook): 464 frames over the seven original fish-model dives. Used to reproduce the
  handoff and for the repair figures (4–7).
- `data/corpus.csv` — **every** rigid-target measurement in prod as of 2026-09-12:
  2,927 frames over 32 dives, extracted by `sql/extract_corpus.sql`. A strict superset of
  `all.csv` (pinned by `tests/test_calibration.py`). **The accuracy numbers come from
  this file.** The 2025 pool dives (480–522) self-calibrate from a checkerboard rather
  than borrowing a slate.
- `data/angles.csv` — the designed foreshortening experiment: one Snook at 0–45° in 5°
  steps, five sessions at two ranges (dives 87/94/103/107/114).

## The physical model, in short

None of the seven August fish-model dives self-calibrates. Each is fish-only and **borrows** a
sibling slate dive's laser extrinsics shot ~30 minutes earlier. In between, the laser
rotates about an effectively fixed pivot — so the borrow error is a **rotation**, not an
arbitrary 4-DOF line change, and of that rotation only the in-plane component $\varphi$
sets metric scale.

$\varphi$ is **invisible to the laser dots**: rotating the axis within the camera–laser
plane moves the projected dot by ~$10^{-13}$ px. The image of a 3-D line fixes only the
plane through the camera centre containing it. That is monocular scale ambiguity, and it
is why no amount of dot-reprojection refinement recovers scale — $\varphi$ must come from
a known length, or from one object at two well-separated ranges.

## Two conventions this notebook enforces

1. **Per-dive error is reported as an angle, not a percentage.** A −8 % dive is not eight
   times worse than a −1 % dive; it is a mount 0.29° off instead of 0.03°, and percent
   error is confounded by each dive's shooting distance. The percent view survives only
   as an appendix figure.
2. **A fish's frames aggregate at $p_{90}$, never a mean.** Foreshortening is one-sided
   negative — an out-of-plane fish reads short, never long — so a mean measures the pose
   distribution rather than the object.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

from fishsense_imwut import calibration as cal, pubfig

pubfig.use_publication_style()

FIGURE_DIR = Path("figures")

# August seven-dive export: reproduces HANDOFF.md and backs the repair figures.
aug_rows = cal.load_rows("data/all.csv")
aug_dives = cal.group_by_dive(aug_rows)
aug_df = cal.to_frame(aug_rows)

# Full corpus: backs every accuracy number.
rows = cal.load_rows("data/corpus.csv")
dives = cal.group_by_dive(rows)
df = cal.to_frame(rows)

# Foreshortening experiment.
angles = cal.load_angles("data/angles.csv")

print(f"August: {len(aug_df)} frames | dives {sorted(aug_dives)}")
print(f"corpus: {len(df)} frames | {len(dives)} dives | models {sorted(df.model_name.unique())}")
print(f"        range {df.depth_m.min():.2f}-{df.depth_m.max():.2f} m")
print(f"angles: {len(angles)} frames | dives {sorted(int(d) for d in angles.dive_id.unique())}")


## 1. Reproduce the handoff before plotting anything

The figures are only worth as much as the port behind them, so the implied-yaw floors are
checked against `HANDOFF.md` §3 verbatim before any figure is drawn. If this cell raises,
the geometry port has drifted and nothing below should be trusted.

In [ ]:
HANDOFF_YAW_FLOORS = {  # HANDOFF.md section 3, verbatim
    58: ({"Grouper": 0.0, "Shark": 0.0}, {"Grouper": 9.0, "Shark": 0.0}),
    59: ({"Grouper": 0.0, "Purple Angel": 0.0, "Shark": 0.0, "Snook": 6.5},
         {"Grouper": 8.8, "Purple Angel": 3.5, "Shark": 0.0, "Snook": 11.9}),
    60: ({"Grouper": 12.5, "Purple Angel": 11.7, "Ruler": 12.6, "Shark": 0.0, "Snook": 17.9},
         {"Grouper": 0.0, "Purple Angel": 0.0, "Ruler": 0.0, "Shark": 0.0, "Snook": 0.0}),
    61: ({"Grouper": 6.4, "Purple Angel": 1.8, "Snook": 8.9},
         {"Grouper": 1.4, "Purple Angel": 0.0, "Snook": 6.3}),
    66: ({"Purple Angel": 9.9, "Shark": 5.4, "Snook": 17.9},
         {"Purple Angel": 0.0, "Shark": 0.0, "Snook": 6.9}),
    76: ({"Grouper": 17.9, "Purple Angel": 11.6, "Shark": 20.0, "Snook": 19.0},
         {"Grouper": 5.8, "Purple Angel": 0.0, "Shark": 2.1, "Snook": 10.7}),
    84: ({"Grouper": 0.0, "Purple Angel": 0.0, "Shark": 0.0, "Snook": 11.1},
         {"Grouper": 3.9, "Purple Angel": 0.9, "Shark": 0.0, "Snook": 12.7}),
}

PHI_DEG = {}   # the phi believed for each dive: anchored where an anchor exists
for did, r in aug_dives.items():
    o, a, inv_k, n = cal.dive_geometry(r)
    anchored = cal.REPAIR_PHI_DEG.get(did, cal.DISPUTED_PHI_DEG.get(did))
    PHI_DEG[did] = (
        anchored if anchored is not None
        else float(np.degrees(cal.fit_phi_joint(r, o, a, inv_k, n)))
    )

for did, (expect_before, expect_after) in HANDOFF_YAW_FLOORS.items():
    r = aug_dives[did]
    o, a, inv_k, n = cal.dive_geometry(r)
    got_before = cal.yaw_floor(r, 0.0, o, a, inv_k, n)
    got_after = cal.yaw_floor(r, np.deg2rad(PHI_DEG[did]), o, a, inv_k, n)
    for label, expect, got in (("before", expect_before, got_before),
                               ("after", expect_after, got_after)):
        assert set(expect) == set(got), f"dive {did} {label}: model set differs"
        for m, v in expect.items():
            assert abs(v - got[m]) < 0.06, f"dive {did} {label} {m}: {v} vs {got[m]:.1f}"

print("HANDOFF section 3 reproduced exactly for all 7 dives")

## 2. Cohorts — one rule for all 32 dives

The August analysis hand-picked its accuracy dives. With 32 dives that is no longer
defensible, so the cohort is **derived**, and the rule lives in `calibration.py`:

1. Take the $p_{90}$ percent error of every (dive, model) cell with ≥ 5 frames.
2. Tukey median polish: cell = overall + **dive effect** + **model effect** + residual.
   The dive effect is the calibration offset; the model effect is the reference / landmark
   offset and follows the model across dives — so a bad reference (Shark) cannot get a dive
   in or out.
3. A dive is accuracy evidence when |dive effect| ≤ `MAX_DIVE_EFFECT_PP` = 2.5 pp.
4. Dives held out **by design**, decided before any corpus number was looked at: the angle
   experiment (single-object dives; the polish cannot separate their pose from Snook), the
   two August repairs (60, 76 — keeping their raw rows would assert the borrow was fine),
   and disputed 66. They still take part in the polish; they just cannot qualify.

Dive 490 is *not* on the design list. It measures its own checkerboard correctly, its
labels are verified on the raw pixels, and every model still reads ~14 % short; the rule
drops it on its own effect (−13.8 pp) and it is reported as unresolved.

The result is pinned as `cal.CORPUS_ACCURACY_DIVES` and by a test, so a change is a diff.


In [ ]:
polish = cal.median_polish(cal.cell_p90_grid(df))
cohort = cal.accuracy_cohort(df)
assert cohort == cal.CORPUS_ACCURACY_DIVES, cohort

accuracy = df[df.dive_id.isin(cohort)]
print(f"accuracy cohort: {len(cohort)} dives, {len(accuracy)} frames -> {cohort}")


def cohort_stats(sel):
    s = df[df.dive_id.isin(sel)]["pct_error"]
    return {"dives": len(set(sel)), "n": len(s), "median %": round(s.median(), 2),
            "p90 %": round(pubfig.nearest_rank_p90(s), 2),
            "mean |err| %": round(s.abs().mean(), 2)}


held_out = set(cal.DESIGN_EXCLUDED_DIVES) | set(cal.UNRESOLVED_DIVES)
candidates = {
    "RULE: |dive effect| <= 2.5 pp": cohort,
    "August five (58/59/60/61/84)": cal.ACCURACY_DIVES,
    "every dive not held out by design": [d for d in dives if d not in held_out],
    "everything except the angle experiment": [d for d in dives if d not in cal.ANGLE_TEST_DIVES],
    "everything": list(dives),
}
summary = pd.DataFrame({k: cohort_stats(v) for k, v in candidates.items()}).T
summary


### Sensitivity to the threshold

The rule has one free number. The cohort membership near the boundary moves with it, the
headline barely does — which is the point of quoting $p_{90}$ over a cohort rather than a
single dive.

In [ ]:
sweep = {}
for thr in (1.5, 2.0, 2.5, 3.0, 3.5):
    sel = cal.accuracy_cohort(df, max_dive_effect_pp=thr)
    sweep[f"|dive effect| <= {thr} pp"] = cohort_stats(sel)
pd.DataFrame(sweep).T


### Every dive, and why it is in or out

`cal_src` is the dive that owns the extrinsics actually used (self = checkerboard
self-calibration; otherwise the borrowed slate dive).

In [ ]:
per_dive = (df.groupby("dive_id")
              .agg(n=("pct_error", "size"),
                   cal_src=("calibration_dive_id", "first"),
                   models=("model_name", lambda s: ",".join(sorted(s.unique()))),
                   median_pct=("pct_error", "median"),
                   p90_pct=("pct_error", pubfig.nearest_rank_p90)))
per_dive["cal_src"] = ["self" if s == d else str(s) for d, s in zip(per_dive.index, per_dive.cal_src)]
per_dive["dive_effect_pp"] = polish.dive_effect.reindex(per_dive.index)


def status(d):
    if d in cal.ANGLE_TEST_DIVES:
        return "held out: angle experiment"
    if d in cal.REPAIR_PHI_DEG:
        return "held out: August repair"
    if d in cal.DISPUTED_DIVES:
        return "held out: disputed"
    if d in cohort:
        return "ACCURACY"
    if np.isnan(per_dive.dive_effect_pp[d]):
        return "no cell with >= 5 frames"
    return "out: |dive effect| > 2.5"


per_dive["status"] = [status(d) for d in per_dive.index]
print(f"polish: overall {polish.overall:+.2f} pp, median |residual| "
      f"{polish.residual.abs().stack().median():.2f} pp")
per_dive.round(2).sort_values("dive_effect_pp")


### Per-model fidelity — the "ladder"

$p_{90}$ per model on the accuracy cohort. Note the $p_{90}$ convention: nearest-rank
`ceil(0.9n)`, matching the `fish_length_estimate` view, so a number quoted from a figure
equals the number the pipeline reports.

The Shark's reference (605 mm) is the one that cannot be re-verified — the model is not
on hand. It is 25 of 842 frames, so it cannot move the headline: dropping it shifts the
cohort $p_{90}$ from +0.02 % to −0.28 %; re-referencing it at 620 mm (the polish implies
~620–626 mm) gives −0.19 %. It is reported with its stated reference and its +1.9 % model effect.

In [ ]:
ladder = accuracy.groupby("model_name").agg(
    known_cm=("known_length_m", lambda s: s.iloc[0] * 100),
    n=("pct_error", "size"),
    median_pct=("pct_error", "median"),
    p90_nearest_rank=("pct_error", pubfig.nearest_rank_p90),
    p90_interpolated=("pct_error", lambda s: np.percentile(s, 90)),
).sort_values("known_cm")

no_shark = accuracy[accuracy.model_name != "Shark"].pct_error
print(f"cohort p90 without Shark: {pubfig.nearest_rank_p90(no_shark):+.2f} %  (n={len(no_shark)})")
ladder.round(2)


## Figure 1 — Measured vs. known length

Accuracy cohort. Six objects separate along the x-axis by their own known lengths, so
identity needs no colour — which keeps this inside the three-series cap that scatter forms
carry. Blue is the per-frame cloud, orange the $p_{90}$ estimator drawn over it.

In [ ]:
fig1 = pubfig.fig_measured_vs_known(accuracy)
pubfig.save_figure(fig1, "fig1_measured_vs_known", FIGURE_DIR)

## Figure 2 — Error distribution by model, with the estimator marked

The box is the frame-level spread; the diamond is the number the pipeline reports. The gap
between the median and the $p_{90}$ **is** the foreshortening tail — this figure is the
argument for the estimator being a high quantile rather than a mean.

In [ ]:
fig2 = pubfig.fig_error_by_model_p90(accuracy)
pubfig.save_figure(fig2, "fig2_error_by_model", FIGURE_DIR)

## Figure 3 — Error vs. range

Accuracy cohort. Triangulation conditioning goes as $Z^2$ for a near-axial laser, so
range-dependence surviving into the delivered measurement would show as a **widening
band** rather than a drifting median.

In [ ]:
fig3 = pubfig.fig_error_vs_depth(accuracy, depth_column="depth_m")
pubfig.save_figure(fig3, "fig3_error_vs_range", FIGURE_DIR)

## Figure 4 — Per-dive mount state $\varphi$

The honest form of "per-dive error". $\varphi$ is anchored where an anchor exists (60 and
66 from the ruler, 76 from its models jointly) and joint-model elsewhere; zero means the
mount did not move between the slate dive and the fish dive.

In [ ]:
cohort_of = {d: "sound" for d in PHI_DEG}
cohort_of.update({d: "repaired" for d in cal.REPAIR_PHI_DEG})
cohort_of.update({d: "disputed" for d in cal.DISPUTED_DIVES})

fig4 = pubfig.fig_phi_mount_state(PHI_DEG, cohort_of, cal.BORROW_MAP)
pubfig.save_figure(fig4, "fig4_phi_mount_state", FIGURE_DIR)

spread = max(PHI_DEG.values()) - min(PHI_DEG.values())
print(f"mount-state phi spans {min(PHI_DEG.values()):+.4f} to {max(PHI_DEG.values()):+.4f} deg")
print(f"spread {spread:.3f} deg  ->  beam misalignment eps >= {spread/2:.3f} deg")

## Figure 5 — The repair, seen through the implied-yaw floor

For each model, the 10th-percentile implied out-of-plane yaw over its frames — the
best-presented ones. Physics gives a hard floor at 0°: you cannot present a rigid object
better than side-on, so a sound calibration with at least one good frame must sit at ~0.

The diagnostic separates the two failure modes by their signature. A **calibration** error
lifts every object's floor **uniformly**; **pose** only adds a one-sided tail *above* the
floor. Dive 60 is the clean case — four independent objects plus the ruler sitting at
11.7–17.9°, all landing at 0.0° on a single fitted parameter.

Read alongside the signed error, always: the floor clips at 0 and is blind to
over-correction, which is exactly how dive 66 slipped through at first.

In [ ]:
panels = []
for did in (60, 76):
    r = aug_dives[did]
    o, a, inv_k, n = cal.dive_geometry(r)
    phi = np.deg2rad(cal.REPAIR_PHI_DEG[did])
    panels.append((
        f"Dive {did} \u2190 {cal.BORROW_MAP[did]}   $\\varphi$ = {cal.REPAIR_PHI_DEG[did]:+.4f}\u00b0",
        cal.yaw_floor(r, 0.0, o, a, inv_k, n),
        cal.yaw_floor(r, phi, o, a, inv_k, n),
    ))

fig5 = pubfig.fig_yaw_floor_repair(panels)
pubfig.save_figure(fig5, "fig5_yaw_floor_repair", FIGURE_DIR)

## Interlude — separating the two error sources

A **calibration** error is per-*dive* and moves every model on that dive together. A
**reference or landmark** error is per-*model* and follows it across every dive. A Tukey
median polish on the $p_{90}$ of each (dive, model) cell separates the two, and the fit is
tight — median $|$residual$|$ 0.31 pp — which is itself evidence that these two additive
terms are the whole story.

This is what the raw ladder cannot show, because the models do not appear in the same
dives.

The polish itself is `cal.median_polish` (it is what the cohort rule runs on); this cell
reads it out for the whole corpus.

In [ ]:
print("MODEL effect (reference / landmark), pp:")
print(polish.model_effect.sort_values().round(2).to_string())
print("\nDIVE effect (calibration), pp:")
print(polish.dive_effect.sort_values().round(2).to_string())
print(f"\nmedian |residual| {polish.residual.abs().stack().median():.2f} pp")

shark_offset = polish.model_effect["Shark"] - polish.model_effect.drop("Shark").median()
print(f"\nShark sits {shark_offset:+.2f} pp above the other models' median effect")
print(f"= {1000 * 0.605 * shark_offset / 100:+.1f} mm on a 605 mm reference")


### Shark's ladder number, decomposed

On the corpus the Shark carries the largest model effect (+3.2 pp) and, like every model,
its ladder $p_{90}$ is the sum of that and whichever dives it happened to be shot on. The
crosstab below shows its coverage: 25 cohort frames across three August dives, none in
the pool sessions. The August analysis's argument that a third of its frames sat on the
most positive dive (58) still holds — 58 is now outside the cohort — but the model effect
is what survives the polish, and it is what a caliper on the Shark would settle.

In [ ]:
coverage = pd.crosstab(df.dive_id, df.model_name)
coverage["dive effect (pp)"] = polish.dive_effect.reindex(coverage.index).round(2)
coverage["in cohort"] = coverage.index.isin(cohort)
coverage


## Figure 6 — The Shark anomaly

Accuracy cohort, Shark accented and the other three recessive. Shark sits entirely
positive while the others straddle or sit below zero. Nothing in the error model pushes a
measurement **long** — foreshortening is one-sided negative — so this points at the
605 mm reference being short, or at labelers including the caudal filament.

It matters beyond curiosity: an anchor's own bias transfers into $\varphi$ at roughly
0.03° per 1 %, so a +4 % anchor biases $\varphi$ by ~0.13°. Grouper is the near-unbiased
anchor and is why it, not Shark, is the one to fit against.

In [ ]:
fig6 = pubfig.fig_error_by_model_p90(accuracy, emphasise="Shark")
pubfig.save_figure(fig6, "fig6_shark_anomaly", FIGURE_DIR)


## Figure 8 — Error vs. fish angle (the foreshortening experiment)

Stage 14 measures the *projection* of the fish onto the image plane at the laser's depth,
so a fish at angle $\theta$ to the plane must read $\cos\theta - 1$ short. The designed
experiment tests exactly that: one Snook (455 mm), stepped 0–45° in 5° increments, five
sessions at ~2 m and ~4.5 m. Each session is a thin line; the pooled median and IQR carry
the result. The gap between the pooled median and the dashed prediction is the broadside
bias (~−3.5 %), not a pose effect. The curve crosses the 15 % budget at 30°.

A sixth session (dive 526, the evening FSL05 burst) is labelled but unmeasured: its slate
frames are a single-distance burst, so its calibration cannot be fitted, and it was parked
on 2026-09-12 rather than measured under a borrowed calibration the experiment itself would
have to validate. See `FINDINGS.md` §7.5.

In [ ]:
fig8 = pubfig.fig_error_vs_angle(angles)
pubfig.save_figure(fig8, "fig8_error_vs_angle", FIGURE_DIR)

pooled = cal.binned_angle_error(angles)
print("pooled median by designed angle:")
print(pooled.round(1).to_string())
first = next((a for a, m in pooled["median"].items() if m < -15), None)
print(f"\nfirst designed angle past the 15 % budget: {first} deg")


## Appendix figure — every dive, percent error

Kept as an appendix so a reader can see the whole corpus and why dives are held out. The
held-out dives are the wide, strongly negative rows: 490 (unresolved), 492/494 (borrowed
calibrations that the rule rejects), 76 and 66 (the August repair / dispute), and the angle
experiment (single object, mostly oblique by design).

In [ ]:
figA = pubfig.fig_error_by_dive(df, min_frames=8, xlim=(-35, 15), figsize=(pubfig.COL_WIDTH, 5.0))
pubfig.save_figure(figA, "figA_all_dives_percent", FIGURE_DIR)


In [ ]:
for path in sorted(FIGURE_DIR.glob("*.pdf")):
    print(path)

### Dropping these into the paper

```latex
\begin{figure}[t]
  \centering
  \includegraphics{figures/fig2_error_by_model.pdf}
  \caption{...}
  \label{fig:error-by-model}
\end{figure}
```

No `width=` — the figures are sized for the `acmart` column already
(`pubfig.COL_WIDTH` 3.33 in; `pubfig.FULL_WIDTH` 7.00 in for a `figure*`, which is what
Figure 5 uses). Scaling in LaTeX rescales the type along with the marks and breaks the
7–8 pt sizing.